# 🩺 Machine Learning — Diagnóstico de Câncer de Mama
**FIAP Postech — IA para Devs | Módulo Machine Learning**

---

## Objetivo
Construir um modelo de classificação para prever se um tumor é **maligno (0)** ou **benigno (1)** com base em características clínicas, utilizando o dataset Wisconsin Breast Cancer.

## Pipeline deste notebook
1. Carregamento e exploração dos dados (EDA)
2. Pré-processamento e Feature Scaling
3. Treinamento e comparação de modelos
4. Avaliação com métricas completas (Recall como principal)
5. Curva ROC e AUC Score
6. Explicabilidade com SHAP

---
> **Por que Recall é a métrica principal?**  
> Em diagnóstico oncológico, um **falso negativo** (classificar maligno como benigno) tem custo muito maior do que um falso positivo. Portanto, minimizamos os falsos negativos maximizando o **Recall**.

## 0. Instalação de dependências

In [1]:
# Execute apenas se necessário
# !pip install shap matplotlib seaborn scikit-learn pandas numpy

## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dataset
from sklearn.datasets import load_breast_cancer

# Pré-processamento
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Métricas
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
    make_scorer,
    recall_score,
    f1_score,
    accuracy_score
)

# Explicabilidade
import shap

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Imports concluídos')

✅ Imports concluídos


---
## 2. Carregamento e Exploração dos Dados (EDA)

In [3]:
# Carregando o dataset
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = maligno, 1 = benigno

print('📊 Shape do dataset:', df.shape)
print('\n🏷️ Classes:', dict(zip([0, 1], data.target_names)))
print('\n📋 Features:', list(data.feature_names))

📊 Shape do dataset: (569, 31)

🏷️ Classes: {0: np.str_('malignant'), 1: np.str_('benign')}

📋 Features: [np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness'), np.str_('mean compactness'), np.str_('mean concavity'), np.str_('mean concave points'), np.str_('mean symmetry'), np.str_('mean fractal dimension'), np.str_('radius error'), np.str_('texture error'), np.str_('perimeter error'), np.str_('area error'), np.str_('smoothness error'), np.str_('compactness error'), np.str_('concavity error'), np.str_('concave points error'), np.str_('symmetry error'), np.str_('fractal dimension error'), np.str_('worst radius'), np.str_('worst texture'), np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst smoothness'), np.str_('worst compactness'), np.str_('worst concavity'), np.str_('worst concave points'), np.str_('worst symmetry'), np.str_('worst fractal dimension')]


In [4]:
# Primeiras linhas
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [5]:
# Informações gerais
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [6]:
# Estatísticas descritivas
df.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


In [7]:
# Verificando valores nulos
nulls = df.isnull().sum()
print('🔍 Valores nulos por coluna:')
print(nulls[nulls > 0] if nulls.sum() > 0 else '✅ Nenhum valor nulo encontrado!')

🔍 Valores nulos por coluna:
✅ Nenhum valor nulo encontrado!


In [8]:
# Distribuição das classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem — sort_index() garante a ordem [0, 1] (value_counts ordena por frequência!)
target_counts = df['target'].value_counts().sort_index()
target_labels = ['Maligno (0)', 'Benigno (1)']
axes[0].bar(target_labels, target_counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição das Classes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Proporção
axes[1].pie(target_counts.values, labels=target_labels,
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporção das Classes', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.suptitle('Balanceamento do Dataset', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print(f'\nRatio Maligno/Benigno: {target_counts[0]}/{target_counts[1]} ({target_counts[0]/len(df)*100:.1f}% / {target_counts[1]/len(df)*100:.1f}%)')


Ratio Maligno/Benigno: 212/357 (37.3% / 62.7%)

In [9]:
# Distribuição das features por classe — primeiras 9 features (mean)
mean_features = [col for col in df.columns if 'mean' in col]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(mean_features[:9]):
    ax = axes[i]
    df[df['target'] == 0][col].hist(ax=ax, alpha=0.6, color='#e74c3c', label='Maligno', bins=25)
    df[df['target'] == 1][col].hist(ax=ax, alpha=0.6, color='#2ecc71', label='Benigno', bins=25)
    ax.set_title(col.replace(' ', '\n'), fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Distribuição das Features (Mean) por Classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [10]:
# Mapa de correlação das features 'mean'
corr_features = df[mean_features + ['target']]

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_features.corr(), dtype=bool))
sns.heatmap(
    corr_features.corr(),
    annot=True, fmt='.2f', cmap='RdYlGn',
    mask=mask, linewidths=0.5,
    vmin=-1, vmax=1, center=0,
    annot_kws={'size': 8}
)
plt.title('Mapa de Correlação — Features Mean + Target', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [11]:
# Correlação com o target — ranking
corr_target = df.corr()['target'].drop('target').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in df.corr()['target'].drop('target').loc[corr_target.index]]
corr_target.plot(kind='barh', color=colors)
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Limiar 0.5')
plt.title('Correlação Absoluta das Features com o Target', fontsize=13, fontweight='bold')
plt.xlabel('|Correlação|')
plt.legend()
plt.tight_layout()
plt.show()

print('\n🔝 Top 5 features mais correlacionadas com o target:')
print(corr_target.head())


🔝 Top 5 features mais correlacionadas com o target:
worst concave points    0.793566
worst perimeter         0.782914
mean concave points     0.776614
worst radius            0.776454
mean perimeter          0.742636
Name: target, dtype: float64


---
## 3. Pré-processamento

In [12]:
# Separando features e target
X = df.drop('target', axis=1)
y = df['target']

# Split estratificado — garante proporção das classes em train e test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ← importante para datasets desbalanceados
)

print(f'Train: {X_train.shape[0]} amostras | Test: {X_test.shape[0]} amostras')
print(f'\nDistribuição no Train:\n{y_train.value_counts()}')
print(f'\nDistribuição no Test:\n{y_test.value_counts()}')

Train: 455 amostras | Test: 114 amostras

Distribuição no Train:
target
1    285
0    170
Name: count, dtype: int64

Distribuição no Test:
target
1    72
0    42
Name: count, dtype: int64


In [13]:
# Scaler — padroniza as features (importante para KNN, SVM, Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # ← fit apenas no train!

# Visualizando o efeito do scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pd.DataFrame(X_train_scaled, columns=X.columns)[mean_features[:5]].boxplot(ax=axes[0])
axes[0].set_title('Após StandardScaler', fontsize=11, fontweight='bold')
axes[0].set_xticklabels([c.replace('mean ', '') for c in mean_features[:5]], rotation=15)

X_train[mean_features[:5]].boxplot(ax=axes[1])
axes[1].set_title('Antes do Scaling', fontsize=11, fontweight='bold')
axes[1].set_xticklabels([c.replace('mean ', '') for c in mean_features[:5]], rotation=15)

plt.suptitle('Efeito do StandardScaler nas Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Treinamento e Comparação de Modelos

> Usamos `Pipeline` do sklearn para garantir que o scaler seja aplicado corretamente dentro do cross-validation (evita data leakage).

In [14]:
# Definindo os pipelines
pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(random_state=42, max_depth=5))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(probability=True, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ])
}

# ⚠️ Classe positiva = MALIGNO (0). O objetivo clínico é minimizar falsos negativos,
# isto é, casos malignos classificados como benignos. Por padrão o sklearn usa
# pos_label=1 (benigno), que mediria o recall da classe ERRADA — por isso fixamos pos_label=0.
recall_maligno = make_scorer(recall_score, pos_label=0)
f1_maligno     = make_scorer(f1_score, pos_label=0)

# Cross-validation estratificada — 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = {}

print('🔄 Rodando Cross-Validation (5 folds) para cada modelo...\n')

for nome, pipeline in pipelines.items():
    scores_recall   = cross_val_score(pipeline, X, y, cv=cv, scoring=recall_maligno)
    scores_f1       = cross_val_score(pipeline, X, y, cv=cv, scoring=f1_maligno)
    scores_accuracy = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    scores_auc      = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc')

    resultados[nome] = {
        'Recall (mean)':   scores_recall.mean(),
        'Recall (std)':    scores_recall.std(),
        'F1 (mean)':       scores_f1.mean(),
        'Accuracy (mean)': scores_accuracy.mean(),
        'AUC (mean)':      scores_auc.mean()
    }

    print(f'✅ {nome}: Recall(maligno)={scores_recall.mean():.3f} ± {scores_recall.std():.3f} | F1(maligno)={scores_f1.mean():.3f} | AUC={scores_auc.mean():.3f}')

df_resultados = pd.DataFrame(resultados).T
print('\n📊 Tabela de Resultados (Recall/F1 referem-se à classe MALIGNA):')
df_resultados.sort_values('Recall (mean)', ascending=False)

🔄 Rodando Cross-Validation (5 folds) para cada modelo...



✅ Logistic Regression: Recall(maligno)=0.944 ± 0.052 | F1(maligno)=0.963 | AUC=0.995


✅ Decision Tree: Recall(maligno)=0.878 ± 0.082 | F1(maligno)=0.899 | AUC=0.907


✅ Random Forest: Recall(maligno)=0.939 ± 0.050 | F1(maligno)=0.941 | AUC=0.989


✅ Gradient Boosting: Recall(maligno)=0.906 ± 0.066 | F1(maligno)=0.929 | AUC=0.993


✅ SVM: Recall(maligno)=0.958 ± 0.037 | F1(maligno)=0.969 | AUC=0.995


✅ KNN: Recall(maligno)=0.925 ± 0.046 | F1(maligno)=0.949 | AUC=0.985

📊 Tabela de Resultados (Recall/F1 referem-se à classe MALIGNA):


,Recall (mean),Recall (std),F1 (mean),Accuracy (mean),AUC (mean)
SVM,0.957586,0.037492,0.968777,0.977162,0.994525
Logistic Regression,0.943632,0.052462,0.963341,0.973669,0.995314
Random Forest,0.938870,0.050328,0.940717,0.956094,0.988548
KNN,0.924585,0.045803,0.948669,0.963096,0.984865
Gradient Boosting,0.906091,0.065707,0.929006,0.949076,0.992731
Decision Tree,0.877852,0.082044,0.899202,0.927993,0.906845


In [15]:
# Visualização comparativa
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metricas_plot = ['Recall (mean)', 'F1 (mean)', 'AUC (mean)']
cores = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

for i, metrica in enumerate(metricas_plot):
    vals = df_resultados[metrica].sort_values(ascending=False)
    axes[i].barh(vals.index, vals.values, color=cores[:len(vals)], edgecolor='white')
    axes[i].set_xlim(0.8, 1.0)
    axes[i].set_title(metrica, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Score')
    for j, (idx, val) in enumerate(vals.items()):
        axes[i].text(val + 0.001, j, f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Comparação de Modelos — Cross-Validation 5-Fold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Avaliação do Melhor Modelo no Test Set

In [16]:
# Selecionando o melhor modelo por Recall
melhor_modelo_nome = df_resultados['Recall (mean)'].idxmax()
print(f'🏆 Melhor modelo por Recall: {melhor_modelo_nome}')

# Treinando no conjunto completo de treino
melhor_pipeline = pipelines[melhor_modelo_nome]
melhor_pipeline.fit(X_train, y_train)

# Predições
y_pred = melhor_pipeline.predict(X_test)
y_proba = melhor_pipeline.predict_proba(X_test)[:, 1]

🏆 Melhor modelo por Recall: SVM


In [17]:
# Classification Report
print('=' * 55)
print(f'  Classification Report — {melhor_modelo_nome}')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno']))

print('\n💡 Interpretação das métricas:')
print('  Precision: dos que o modelo marcou como positivo, quantos realmente são?')
print('  Recall:    dos que realmente são positivos, quantos o modelo encontrou?')
print('  F1-Score:  média harmônica entre Precision e Recall')
print('  Support:   número de amostras reais de cada classe no test set')

  Classification Report — SVM
              precision    recall  f1-score   support

     Maligno       0.98      0.98      0.98        42
     Benigno       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


💡 Interpretação das métricas:
  Precision: dos que o modelo marcou como positivo, quantos realmente são?
  Recall:    dos que realmente são positivos, quantos o modelo encontrou?
  F1-Score:  média harmônica entre Precision e Recall
  Support:   número de amostras reais de cada classe no test set


In [18]:
# Matriz de Confusão
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Valores absolutos
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Maligno', 'Benigno'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Matriz de Confusão\n{melhor_modelo_nome}', fontweight='bold')

# Normalizada
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=['Maligno', 'Benigno'])
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f'Matriz de Confusão Normalizada\n{melhor_modelo_nome}', fontweight='bold')

plt.tight_layout()
plt.show()

# Destacando falsos negativos (classe positiva = MALIGNO/0; labels ordenados [0=maligno, 1=benigno])
fn = cm[0][1]  # real=maligno, pred=benigno  → Falso Negativo (erro crítico)
fp = cm[1][0]  # real=benigno, pred=maligno  → Falso Positivo
print(f'\n⚠️  Falsos Negativos (maligno classificado como benigno): {fn}')
print(f'   Falsos Positivos (benigno classificado como maligno): {fp}')
print(f'\n   Em contexto clínico, FN é o erro mais crítico.')


⚠️  Falsos Negativos (maligno classificado como benigno): 1
   Falsos Positivos (benigno classificado como maligno): 1

   Em contexto clínico, FN é o erro mais crítico.


In [19]:
# Curva ROC e AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Classifier (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.1, color='#3498db')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Especificidade)', fontsize=11)
plt.ylabel('True Positive Rate (Recall / Sensibilidade)', fontsize=11)
plt.title(f'Curva ROC — {melhor_modelo_nome}', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n🎯 AUC Score: {auc_score:.4f}')
print('   AUC = 1.0 → classificação perfeita')
print('   AUC = 0.5 → equivale a chute aleatório')


🎯 AUC Score: 0.9950


   AUC = 1.0 → classificação perfeita
   AUC = 0.5 → equivale a chute aleatório


In [20]:
# Curvas ROC de todos os modelos sobrepostas
plt.figure(figsize=(9, 7))
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC=0.50)')

for nome, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    fpr_m, tpr_m, _ = roc_curve(y_test, proba)
    auc_m = roc_auc_score(y_test, proba)
    plt.plot(fpr_m, tpr_m, lw=2, label=f'{nome} (AUC={auc_m:.3f})')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Comparação de Curvas ROC — Todos os Modelos', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Explicabilidade com SHAP

SHAP (SHapley Additive exPlanations) quantifica a contribuição de cada feature para cada predição individual, baseado na teoria dos jogos cooperativos (Shapley values).

In [21]:
# Usando Random Forest para SHAP (Tree-based)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Criando o explainer
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# Compatibilidade entre versões do SHAP:
#   - antigas:        lista [classe0, classe1], cada item (n_amostras, n_features)
#   - novas (>=0.43): ndarray (n_amostras, n_features, n_classes)
# Selecionamos sempre a classe benigna (1).
if isinstance(shap_values, list):
    shap_values_benigno = shap_values[1]
else:
    shap_values_benigno = shap_values[..., 1]

print('✅ SHAP values calculados')
print(f'   Shape bruto: {np.array(shap_values).shape}')
print(f'   Shape usado (classe benigna): {np.array(shap_values_benigno).shape}  →  [amostras, features]')

✅ SHAP values calculados
   Shape bruto: (114, 30, 2)
   Shape usado (classe benigna): (114, 30)  →  [amostras, features]


In [22]:
# Summary Plot — importância global das features
shap.summary_plot(
    shap_values_benigno,  # classe 1 = benigno
    X_test,
    feature_names=list(X.columns),
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP — Importância Global das Features (Top 15)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [23]:
# Beeswarm plot — distribuição dos SHAP values por feature
shap.summary_plot(
    shap_values_benigno,
    X_test,
    feature_names=list(X.columns),
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm — Impacto e Direção das Features', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\n💡 Leitura do gráfico:')
print('   Cor vermelha  → valor alto da feature')
print('   Cor azul      → valor baixo da feature')
print('   Posição X     → impacto na predição (+ = empurra para benigno)')


💡 Leitura do gráfico:
   Cor vermelha  → valor alto da feature
   Cor azul      → valor baixo da feature
   Posição X     → impacto na predição (+ = empurra para benigno)


In [24]:
# Force plot — explicando uma predição individual
idx = 0  # Primeira amostra do test set

print(f'📍 Explicando amostra #{idx}')
print(f'   Real:     {"Benigno" if y_test.iloc[idx] == 1 else "Maligno"}')
print(f'   Predito:  {"Benigno" if rf_model.predict(X_test.iloc[[idx]])[0] == 1 else "Maligno"}')
print(f'   Probabilidade (benigno): {rf_model.predict_proba(X_test.iloc[[idx]])[0][1]:.3f}')

shap.force_plot(
    explainer.expected_value[1],
    shap_values_benigno[idx],
    X_test.iloc[idx],
    feature_names=list(X.columns),
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot — Amostra #{idx}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

📍 Explicando amostra #0
   Real:     Maligno
   Predito:  Maligno
   Probabilidade (benigno): 0.000


In [25]:
# Feature Importance comparando: RF nativo vs SHAP
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
shap_importance = pd.Series(np.abs(shap_values_benigno).mean(axis=0), index=X.columns)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rf_importance.nlargest(15).sort_values().plot(kind='barh', ax=axes[0], color='#3498db')
axes[0].set_title('Feature Importance\n(Random Forest nativo)', fontweight='bold')
axes[0].set_xlabel('Importância')

shap_importance.nlargest(15).sort_values().plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Feature Importance\n(SHAP mean |Shapley value|)', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|')

plt.suptitle('Comparação: Importância RF vs SHAP', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Resumo Executivo

In [26]:
print('=' * 60)
print('           RESUMO DO ESTUDO — FIAP ML Challenge')
print('=' * 60)
print(f'\n📦 Dataset: Wisconsin Breast Cancer')
print(f'   {len(df)} amostras | 30 features | 2 classes')
print(f'   Maligno: {(y==0).sum()} | Benigno: {(y==1).sum()}')
print(f'\n🎯 Métrica principal: Recall da classe MALIGNA (pos_label=0)')
print(f'   Justificativa: minimizar falsos negativos em diagnóstico oncológico')
print(f'\n📊 Comparação dos modelos (Cross-Validation 5-Fold):')
print(df_resultados[['Recall (mean)', 'F1 (mean)', 'AUC (mean)']]
      .sort_values('Recall (mean)', ascending=False)
      .to_string())

# Avaliando no test set
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(f'\n🏆 Modelo final avaliado no test set: Random Forest')
print(f'   Recall (maligno):   {recall_score(y_test, y_pred_rf, pos_label=0):.4f}')
print(f'   F1-Score (maligno): {f1_score(y_test, y_pred_rf, pos_label=0):.4f}')
print(f'   Accuracy:           {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'   AUC:                {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'\n🔍 Explicabilidade: SHAP (TreeExplainer)')
top3_shap = shap_importance.nlargest(3).index.tolist()
print(f'   Top 3 features mais influentes:')
for i, feat in enumerate(top3_shap, 1):
    print(f'   {i}. {feat}')
print('\n' + '=' * 60)

           RESUMO DO ESTUDO — FIAP ML Challenge

📦 Dataset: Wisconsin Breast Cancer
   569 amostras | 30 features | 2 classes
   Maligno: 212 | Benigno: 357

🎯 Métrica principal: Recall da classe MALIGNA (pos_label=0)
   Justificativa: minimizar falsos negativos em diagnóstico oncológico

📊 Comparação dos modelos (Cross-Validation 5-Fold):
                     Recall (mean)  F1 (mean)  AUC (mean)
SVM                       0.957586   0.968777    0.994525
Logistic Regression       0.943632   0.963341    0.995314
Random Forest             0.938870   0.940717    0.988548
KNN                       0.924585   0.948669    0.984865
Gradient Boosting         0.906091   0.929006    0.992731
Decision Tree             0.877852   0.899202    0.906845

🏆 Modelo final avaliado no test set: Random Forest
   Recall (maligno):   0.9286
   F1-Score (maligno): 0.9398
   Accuracy:           0.9561
   AUC:                0.9937

🔍 Explicabilidade: SHAP (TreeExplainer)
   Top 3 features mais influentes:
   1

---
## 8. Próximos Passos

Para aprofundar e ir além do Tech Challenge:

| Técnica | Por que explorar? |
|---|---|
| **GridSearchCV / RandomizedSearchCV** | Otimização de hiperparâmetros dos modelos |
| **SMOTE** | Técnica de oversampling para datasets desbalanceados |
| **Threshold tuning** | Ajustar o limiar de decisão para maximizar Recall |
| **Redução de dimensionalidade (PCA)** | Visualizar clusters e reduzir 30→2 features |
| **LIME** | Alternativa ao SHAP para explicabilidade local |
| **MLflow** | Registro e rastreamento de experimentos |

---
*Notebook criado para FIAP Postech — IA para Devs | Módulo Machine Learning*